# Database Overview Notebook

This notebook connects to the PostgreSQL database used by the project, inspects how much market data is stored, and shows how to resume data pulling without duplication.

In [2]:
# Ensure the project root is on the Python path so that `db` can be imported
import sys, os
project_root = os.path.abspath('..')  # notebook is in `notebooks/`
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from db.connection import ConnectionManager
import pandas as pd

In [4]:
# List tables in the public schema
mgr = ConnectionManager()
conn = mgr.get_connection()
cur = conn.cursor()
cur.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';")
tables = [row[0] for row in cur.fetchall()]
print('Tables:', tables)

Tables: ['ohlcv_1min', 'ohlcv_5min', 'instruments', 'tick_data', 'download_log']


In [5]:
# Count rows in key tables (ticks, ohlcv, instruments)
for tbl in tables:
    cur.execute(f"SELECT COUNT(*) FROM {tbl};")
    count = cur.fetchone()[0]
    print(f'{tbl}: {count:,} rows')

ohlcv_1min: 1,168,428 rows
ohlcv_5min: 316,621 rows
instruments: 3,667 rows
tick_data: 3,098,543 rows
download_log: 1,173 rows


In [7]:
# Determine the most recent entry per instrument (for resuming)
cur.execute("SELECT * FROM tick_data limit 10;")
latest = cur.fetchall()
latest
df_latest = pd.DataFrame(latest)
# display(df_latest.head())

In [10]:
type(latest)

list

In [8]:
df_latest

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,219,2503,2026-03-05 09:12:13.520052,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:22.514967
1,220,2503,2026-03-05 09:12:14.513005,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:22.514967
2,221,2503,2026-03-05 09:12:15.512010,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:22.514967
3,222,2503,2026-03-05 09:12:16.512986,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:22.514967
4,223,2503,2026-03-05 09:12:17.515459,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:22.514967
5,224,2503,2026-03-05 09:12:18.513276,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:22.514967
6,225,2503,2026-03-05 09:12:19.513112,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:22.514967
7,226,2503,2026-03-05 09:12:20.512110,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:22.514967
8,227,2503,2026-03-05 09:12:21.513365,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:22.514967
9,228,2503,2026-03-05 09:12:22.601911,24615.95,0,24615.95,24615.95,24615.95,24480.50,None,0,0,0,0,"[{'price': -0.01, 'orders': None, 'quantity': ...",None,2026-03-05 09:12:26.513315


## Resuming Data Pull

The downloader functions (e.g., `download_historical_data` in `downloader/ohlcv.py`) accept a `start_date`/`end_date` parameter.

To resume safely you can:
1. Query the latest timestamp for each instrument as shown above.
2. Choose the newest timestamp across all instruments (or per‑instrument) and add a small buffer (e.g., 1 minute).
3. Call the downloader with `start_date` set to that buffered timestamp.

```python
from downloader.ohlcv import download_historical_data

# Example: resume from the most recent tick across all instruments
latest_ts = df_latest['last_timestamp'].max()
# add a minute to avoid overlap
resume_date = (latest_ts + pd.Timedelta(minutes=1)).strftime('%Y-%m-%d')

# Provide the desired instrument types
instrument_types = ['AMXIDX', 'FUTIDX', 'OPTIDX']

# Pull data from the resume date onward (e.g., next 30 days)
download_historical_data(instrument_types=instrument_types, start_date=resume_date, days=30)
```

Running the above cell will fetch any missing data without duplicating existing rows.